# Stochastic Calculus, Step by Step · Lesson 31: Linear algebra for several risks

Companion notebook for the lesson. Every number in the lesson's text, tables and figures is produced by running this notebook and read from `build/outputs.json`.

> **The course's numbers.** A share priced at S₀ = 100, a strike K = 105, a risk-free rate r = 3% (continuously compounded), a volatility σ = 20% and a horizon T = 1 year. Where a lesson needs the share's real-world expected return, it is μ = 8%. Made up for teaching; every random number is simulated with a fixed seed.
>
> Educational material only, not financial advice.

1. Setup · 2. Portfolios and the covariance matrix · 3. Which correlation tables are possible · 4. Cholesky: correlated draws · 5. Eigenvalues and PCA · 6. The five-share basket again · 7. Exercises · 8. Export

**Running in Google Colab?** Nothing to install: this notebook only needs numpy, pandas and matplotlib, which Colab already has. Run the cells in order.

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"
seed = 2026

## 1. Setup

In [ ]:
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

S0, K, r, sigma, T = 100.0, 105.0, 0.03, 0.20, 1.0     # the course's numbers
N = lambda x: 0.5 * (1 + math.erf(x / math.sqrt(2)))   # standard normal distribution function
out = {"meta": {
    "series": "Stochastic Calculus, Step by Step",
    "lesson": 31,
    "quantlib_version": "not used",
    "quotes_label": "simulated (no market data)",
    "seed": seed,
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}, "inputs": {"S0": S0, "K": K, "r": r, "sigma": sigma, "T": T, "r_pct": r * 100, "sigma_pct": sigma * 100}}

In [ ]:
mu = 0.08                         # the share's real-world expected return
theta_mpr = (mu - r) / sigma      # the market price of risk
Nv = np.vectorize(N)              # N, for arrays
phi = lambda x: np.exp(-np.asarray(x, float) ** 2 / 2) / math.sqrt(2 * math.pi)   # the normal density


def d_plus(s, tau, sig=sigma, k=K):
    return (np.log(s / k) + (r + 0.5 * sig**2) * tau) / (sig * np.sqrt(tau))


def d_minus(s, tau, sig=sigma, k=K):
    return d_plus(s, tau, sig, k) - sig * np.sqrt(tau)


def call(s, tau, sig=sigma, k=K):
    """The Black-Scholes call price: Lesson 20 derives it."""
    return s * Nv(d_plus(s, tau, sig, k)) - k * np.exp(-r * tau) * Nv(d_minus(s, tau, sig, k))


def put(s, tau, sig=sigma, k=K):
    return call(s, tau, sig, k) - s + k * np.exp(-r * tau)


out["bs"] = {"mu": mu, "theta": theta_mpr, "d_plus": float(d_plus(S0, T)), "d_minus": float(d_minus(S0, T)),
             "N_plus": N(float(d_plus(S0, T))), "N_minus": N(float(d_minus(S0, T))),
             "call": float(call(S0, T)), "put": float(put(S0, T))}
out["bs"]

## 2. Portfolios and the covariance matrix

Two shares: the course's share (σ₁ = 20%) and Lesson 27's second share (σ₂ = 30%), with correlation ρ = 0.5. Hold half of your money in each. The portfolio's variance is $w^\top\Sigma w$, where $\Sigma$ is the covariance matrix. We check it against simulated returns.

In [ ]:
s1, s2, rho = 0.20, 0.30, 0.5
Sigma2 = np.array([[s1**2, rho * s1 * s2], [rho * s1 * s2, s2**2]])
w2 = np.array([0.5, 0.5])
var_w = float(w2 @ Sigma2 @ w2)
# by hand: w1^2 s1^2 + w2^2 s2^2 + 2 w1 w2 rho s1 s2
terms = [0.25 * s1**2, 0.25 * s2**2, 2 * 0.25 * rho * s1 * s2]
# simulate correlated returns with Cholesky (section 4 explains it) and measure
rng = np.random.default_rng(seed)
L2 = np.linalg.cholesky(Sigma2)
R = rng.standard_normal((200_000, 2)) @ L2.T
port = R @ w2
# the minimum-variance mix
w_min = (s2**2 - rho * s1 * s2) / (s1**2 + s2**2 - 2 * rho * s1 * s2)
var_min = float(np.array([w_min, 1 - w_min]) @ Sigma2 @ np.array([w_min, 1 - w_min]))
out["port"] = {"s1": s1, "s2": s2, "rho": rho, "Sigma": Sigma2.tolist(), "var": var_w, "sd": math.sqrt(var_w), "terms": terms,
               "sim_var": float(port.var()), "sim_corr": float(np.corrcoef(R.T)[0, 1]), "w_min": w_min, "var_min": var_min,
               "sd_min": math.sqrt(var_min), "naive_sd": 0.5 * (s1 + s2)}
out["port"]

## 3. Which correlation tables are possible

A variance can't be negative, so $w^\top\Sigma w \ge 0$ for every $w$: the matrix must be **positive semi-definite**. Here is a table that looks reasonable entry by entry and isn't: shares 1 and 2 move together, 2 and 3 move together, but 1 and 3 move opposite.

In [ ]:
C_bad = np.array([[1.0, 0.9, -0.9], [0.9, 1.0, 0.9], [-0.9, 0.9, 1.0]])
w_bad = np.array([1.0, -1.0, 1.0])
q_bad = float(w_bad @ C_bad @ w_bad)
ev_bad, V_bad = np.linalg.eigh(C_bad)
det_bad = float(np.linalg.det(C_bad))
# the range of rho13 that makes the table possible, with rho12 = rho23 = 0.9: det >= 0
a_ = 0.9
roots = sorted(np.roots([1.0, -2 * a_ * a_, 2 * a_ * a_ - 1]).real)
out["bad"] = {"q": q_bad, "eig": ev_bad.tolist(), "det": det_bad, "w_worst": (V_bad[:, 0] / np.abs(V_bad[:, 0]).max()).tolist(),
              "rho13_low": float(roots[0]), "rho13_high": float(roots[1])}
out["bad"]

## 4. Cholesky: correlated draws from independent ones

Write $\Sigma = LL^\top$ with $L$ lower triangular. Then $X = LZ$ with independent standard normals $Z$ has covariance $LL^\top = \Sigma$. The recipe fills $L$ one column at a time. Written out by hand, then checked against numpy, on a three-share example:

In [ ]:
def cholesky(A):
    # Lower-triangular L with L @ L.T = A, column by column. Fails (raises) if A is not positive definite.
    n = len(A)
    L = np.zeros((n, n))
    for j in range(n):
        d = A[j, j] - L[j, :j] @ L[j, :j]
        if d <= 0:
            raise ValueError(f"not positive definite: pivot {j + 1} is {d:.4f}")
        L[j, j] = math.sqrt(d)
        for i in range(j + 1, n):
            L[i, j] = (A[i, j] - L[i, :j] @ L[j, :j]) / L[j, j]
    return L


L_hand = cholesky(Sigma2)
vols3 = np.array([0.20, 0.30, 0.25])
C3 = np.array([[1.0, 0.5, 0.3], [0.5, 1.0, 0.4], [0.3, 0.4, 1.0]])
Sigma3 = np.outer(vols3, vols3) * C3
L3 = cholesky(C3)
try:
    cholesky(C_bad)
    fail = ""
except ValueError as e:
    fail = str(e)
rng = np.random.default_rng(seed + 1)
X3 = rng.standard_normal((400_000, 3)) @ (cholesky(Sigma3)).T
out["chol"] = {"L2": L_hand.tolist(), "L2_numpy_gap": float(np.abs(L_hand - np.linalg.cholesky(Sigma2)).max()),
               "C3": C3.tolist(), "L3": L3.tolist(), "L3_gap": float(np.abs(L3 @ L3.T - C3).max()),
               "sim_cov_gap": float(np.abs(np.cov(X3.T) - Sigma3).max()), "bad_fail": fail}
# draws for the figure: X2 = rho Z1 + sqrt(1 - rho^2) Z2
rng = np.random.default_rng(seed + 2)
Zf = rng.standard_normal((600, 2))
out["chol"]["fig"] = {"z1": Zf[:, 0].round(3).tolist(),
                      **{f"x2_{k}": (rho_ * Zf[:, 0] + math.sqrt(1 - rho_**2) * Zf[:, 1]).round(3).tolist()
                         for k, rho_ in (("m05", -0.5), ("0", 0.0), ("09", 0.9))}}
pd.DataFrame(L3, columns=["col 1", "col 2", "col 3"]).round(4)

## 5. Eigenvalues and PCA

Write $\Sigma = Q\Lambda Q^\top$: $Q$'s columns are the eigenvectors (directions), $\Lambda$ the eigenvalues (variances along them). For five shares with equal correlation ρ = 0.5, the first direction is "everything together", with eigenvalue $1 + 4\rho = 3$: 60% of the total. Estimated from a year of simulated daily returns, the eigenvalues come out close.

In [ ]:
n5, rho5 = 5, 0.5
C5 = np.full((n5, n5), rho5) + (1 - rho5) * np.eye(n5)
ev5, Q5 = np.linalg.eigh(C5)
ev5, Q5 = ev5[::-1], Q5[:, ::-1]
rng = np.random.default_rng(seed + 3)
daily = rng.standard_normal((252, n5)) @ cholesky(C5).T
ev_hat = np.sort(np.linalg.eigvalsh(np.corrcoef(daily.T)))[::-1]
# repairing the impossible table: clip negative eigenvalues, then put ones back on the diagonal
lam, V = np.linalg.eigh(C_bad)
fixed = V @ np.diag(np.maximum(lam, 1e-6)) @ V.T
dsq = np.sqrt(np.diag(fixed))
fixed = fixed / np.outer(dsq, dsq)
out["pca"] = {"eig": ev5.tolist(), "share1": float(ev5[0] / ev5.sum()), "v1": Q5[:, 0].tolist(), "eig_hat": ev_hat.tolist(),
              "share_hat": float(ev_hat[0] / ev_hat.sum()), "fixed": fixed.round(4).tolist(),
              "fixed_eig_min": float(np.linalg.eigvalsh(fixed).min())}
pd.DataFrame({"exact": ev5, "from 252 days": ev_hat}).round(3)

## 6. The five-share basket again

Lesson 30 priced a call on the average of five shares, each with σ = 20% and pairwise correlation 0.5, using numpy's Cholesky. With the hand-written one and the same random numbers, the price is the same to every digit. The basket's own volatility is $\sqrt{w^\top\Sigma w}$.

In [ ]:
w5 = np.full(n5, 1 / n5)
Sigma5 = sigma**2 * C5
vol_basket = math.sqrt(w5 @ Sigma5 @ w5)
gen = np.random.default_rng(seed + 2)        # Lesson 30's generator and draws
Zb = gen.standard_normal((1_000_000, n5)) @ cholesky(C5).T
S_b = S0 * np.exp((r - 0.5 * sigma**2) * T + sigma * math.sqrt(T) * Zb)
pay_b = math.exp(-r * T) * np.maximum(S_b.mean(axis=1) - K, 0)
out["basket"] = {"vol": vol_basket, "vol_formula": sigma * math.sqrt((1 + (n5 - 1) * rho5) / n5), "price": float(pay_b.mean()),
                 "se": float(pay_b.std() / 1000), "lesson30": 5.339103666066505, "call_at_vol": float(call(S0, T, vol_basket)),
                 "vol_limit": sigma * math.sqrt(rho5)}
out["basket"]

In [ ]:
# Chart data for the video (no random numbers)
n_fig = np.arange(1, 41)
out["basket"]["fig"] = {"n": n_fig.tolist(), "vol": (100 * sigma * np.sqrt((1 + (n_fig - 1) * rho5) / n_fig)).round(4).tolist(),
                        "limit": [round(100 * sigma * math.sqrt(rho5), 4)] * len(n_fig),
                        "ticks": [{"x": float(v), "label": f"{v}"} for v in (1, 10, 20, 30, 40)]}
out["pca"]["rows"] = [{"k": i + 1, "eig": out["pca"]["eig"][i], "eig_hat": out["pca"]["eig_hat"][i]} for i in range(5)]

## 7. Exercises

Try each one before looking anything up. Full solutions are in the companion volume, *Part 5: Solutions*.

**Exercise 1.** Two shares have volatilities 20% and 30% and correlation 0.5. Find the variance and volatility of a portfolio with half its money in each, term by term. Then find the weights that make the variance as small as possible, and that smallest volatility.

**Exercise 2.** Show that if $Z$ is a vector of independent standard normals and $X = LZ$, then $\mathrm{Cov}(X) = LL^\top$. Find the Cholesky factor $L$ of the covariance matrix of the two shares in Exercise 31.1 by matching entries, and check $LL^\top$.

**Exercise 3.** Show that the correlation table with $\rho_{12} = \rho_{23} = 0.9$ and $\rho_{13} = -0.9$ is impossible, using the weights $w = (1, -1, 1)$. Keeping $\rho_{12} = \rho_{23} = 0.9$, for which values of $\rho_{13}$ is the table possible? (Use the determinant.)

**Exercise 4.** Let $C$ be the $n \times n$ matrix with ones on the diagonal and $\rho$ everywhere else. Show that the vector of ones is an eigenvector with eigenvalue $1 + (n - 1)\rho$, and that every vector whose entries add up to zero is an eigenvector with eigenvalue $1 - \rho$. What is the smallest $\rho$ that $n$ shares can all share?

**Exercise 5.** Find the Cholesky factor of the correlation matrix with $\rho_{12} = 0.5$, $\rho_{13} = 0.3$ and $\rho_{23} = 0.4$ by hand, one column at a time.

**Exercise 6.** An equal-weighted basket holds $n$ shares, each with volatility $\sigma$ and every pair with correlation $\rho$. Show that its volatility is $\sigma\sqrt{(1 + (n - 1)\rho)/n}$. Evaluate it for $n = 5$, $\rho = 0.5$, $\sigma = 20\%$, and find its limit as $n \to \infty$. What does the limit say about diversification?

**Exercise 7.** (Code) Write a Cholesky routine that reports which pivot fails when the matrix isn't positive definite. Test it on the matrices of this lesson, and check it against numpy's on 1000 random correlation matrices.

**Exercise 8.** (Code) Price the five-share basket call for $\rho = 0, 0.25, 0.5, 0.75, 1$ by Monte Carlo, and compare each with a Black-Scholes call at the basket's volatility $\sigma\sqrt{(1 + 4\rho)/5}$. Where is the approximation best, and why is it exact at $\rho = 1$?

## 8. Export

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")